## Import modules

In [1]:
import csv
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sn  #Per heatmap
import time
import scipy as sp
import os

Audio-specific modules

In [2]:
import librosa as lb
import librosa.display

Sklearn modules

In [92]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, LabelEncoder,OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, ParameterGrid, train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor

## Load data

In [138]:
data = pd.read_csv('../data/development.csv', index_col=0) #### USA SEMPRE .././data/NOME_FILE o .././data/NOME_CARTELLA per accedere ai dati

In [139]:
data['path'] = data['path'].map(lambda x: x.split('/')[1])
#data = data.set_index('path')
display(data)

,sampling_rate,age,gender,ethnicity,mean_pitch,max_pitch,min_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,num_words,num_characters,num_pauses,silence_duration
path,,,,,,,,,,,,,,,,,,
1.wav,22050,24.0,female,arabic,1821.69060,3999.7170,145.43066,0.013795,0.082725,0.002254,0.210093,3112.257251,[151.99908088],-123.999726,69,281,39,23.846893
2.wav,22050,22.5,female,hungarian,1297.81870,3998.8590,145.37268,0.025349,0.096242,0.007819,0.078849,1688.016389,[129.19921875],-86.928478,69,281,21,19.388662
3.wav,22050,22.0,female,portuguese,1332.85240,3998.8025,145.42395,0.019067,0.119456,0.002974,0.105365,2576.901706,[117.45383523],-98.450670,69,281,1,21.640998
4.wav,22050,22.0,female,english,1430.34990,3998.4510,147.98083,0.017004,0.102389,0.022371,0.173701,3269.751413,[117.45383523],-56.459762,69,281,9,19.644127
5.wav,22050,22.0,male,dutch,1688.72340,3998.6113,145.44772,0.028027,0.124831,0.005369,0.107279,1930.897375,[112.34714674],-80.349204,69,281,11,18.041905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2929.wav,22050,24.0,male,english,1641.14930,3999.1616,145.39359,0.023647,0.115361,0.001879,0.111799,2188.853478,[184.5703125],-100.921055,69,281,11,17.461406
2930.wav,22050,15.0,female,igbo,1089.60050,3984.6550,145.58409,0.015317,0.126740,0.000339,0.070508,2712.362323,[83.35433468],6.757283,0,0,1,1.509206
2931.wav,22050,17.0,female,igbo,994.46484,3989.1785,148.97475,0.009677,0.103535,0.001464,0.058442,2248.698477,[89.10290948],-53.913449,1,9,1,1.645034


## Data Extraction

In [140]:
def load_data(audio_files, folder_path, sample_rate):
    time_stamps = []
    audio_arrays = {}
    mfccs = {}
    for file_name in audio_files:
        if file_name.endswith(".wav"):
            audio_array = lb.load(folder_path + file_name)
            trimmed_audio_array= lb.effects.trim(audio_array[0])
            time_stamps.append(len(trimmed_audio_array[0]))
            audio_arrays[file_name] = trimmed_audio_array[0]
            mfccs[file_name] = lb.feature.mfcc(y=trimmed_audio_array[0], sr=sample_rate, n_mfcc=15, n_fft=2048,hop_length=512)
    return (audio_arrays, mfccs, np.array(time_stamps))

In [141]:
def standardize_time_mfcc(mfcc,num_buckets):
    reduced_list =[]
    existing_int = mfcc.shape[1]
    for i in range(0,existing_int-(existing_int%num_buckets),existing_int//num_buckets):
        reduced_list.append(np.mean(mfcc[:, i:i+(mfcc.shape[1]//num_buckets)], axis=1).flatten())
    return np.array(reduced_list).transpose()

In [142]:
def standardize_all(mfcc_dict,num_buckets):
    standard_mfccs = {}
    for name, mfcc in mfcc_dict.items():
        standard_mfccs[name] = standardize_time_mfcc(mfcc,num_buckets)
    return standard_mfccs

In [143]:
def reshape_all(mfcc_dict):
    reshaped_mfccs = {}
    for name, mfcc in mfcc_dict.items():
        reshaped_mfccs[name] = mfcc.flatten()
    return pd.DataFrame(reshaped_mfccs).T

### Main

In [144]:
SAMPLE_RATE = 22050
num_time_buckets = 20

In [ ]:
audio_files = os.listdir('../data/audios_development')
audio_arrays,mfccs,time_sampl = load_data(audio_files, '../data/audios_development/', SAMPLE_RATE)

In [83]:
standardized_mfccs = standardize_all(mfccs, num_time_buckets)

In [89]:
reshaped_mfccs = reshape_all(standardized_mfccs)

In [90]:
display(reshaped_mfccs)

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
1.wav,-423.020538,-332.887329,-361.605164,-347.482208,-389.083527,-328.815033,-398.173309,-336.800690,-379.510284,-345.724213,...,-1.942417,-5.246557,-4.376464,0.120627,-5.158453,-1.831522,-3.805745,-3.881568,-4.000273,-8.090446
10.wav,-464.710785,-267.225800,-201.898865,-216.123856,-173.217758,-220.823898,-91.316536,-163.099533,-190.114258,-243.981277,...,4.086848,8.919844,12.513921,-1.471938,-5.042432,-2.831152,-1.790807,0.321547,1.039343,-6.841827
100.wav,-318.136597,-421.639404,-281.229004,-350.477203,-380.836121,-313.451111,-382.842468,-272.421265,-430.260315,-280.430054,...,-0.412136,2.075336,6.061966,6.569131,4.929160,3.960616,9.124262,10.854587,7.318303,9.405965
1000.wav,-396.088837,-338.936218,-441.260010,-421.752289,-436.263184,-393.596039,-496.009613,-445.108246,-517.327332,-557.111511,...,-0.204290,-18.454632,-7.708149,2.199664,-3.484957,-9.490080,-12.075005,-3.242330,-2.788409,-8.915504
1001.wav,-399.899261,-363.629517,-339.692078,-331.780243,-292.952393,-279.624115,-329.839478,-323.489655,-297.451721,-304.272430,...,-0.337629,-3.771065,-12.894007,-20.165813,-24.442867,-14.174511,-6.639238,-14.892796,-18.004873,-17.359259
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995.wav,-237.101822,-251.029114,-225.179825,-240.902634,-214.896286,-328.902344,-288.886658,-309.547485,-216.283569,-249.384491,...,-0.380812,5.354548,-7.917774,-6.913934,-12.418830,-6.988984,0.467288,-3.999441,4.249442,-0.261314
996.wav,-260.329346,-303.837067,-268.231964,-296.425598,-339.485199,-265.379089,-306.549103,-267.765625,-334.914368,-252.238235,...,1.907920,4.307010,6.263086,4.819046,6.840928,0.700918,4.721804,7.176890,6.416972,4.897392
997.wav,-253.112915,-137.904434,-214.022568,-279.082306,-233.791412,-256.458740,-281.646332,-233.430222,-238.886230,-231.248749,...,-15.474440,-9.980843,-12.830086,-9.290381,-5.878903,-2.367623,-8.148173,-0.053261,0.816683,-4.935268
998.wav,-396.649658,-330.637207,-297.137207,-334.702545,-309.985260,-382.760620,-281.101410,-342.125122,-270.877258,-336.018799,...,-7.688023,-12.178954,-8.236415,-5.629763,-8.098529,-6.090665,-4.651963,-11.114296,-4.836049,-4.849733


## Modelli

In [91]:
for col,imp in sorted(zip(reshaped_mfccs.columns, RandomForestRegressor(n_jobs=-1).fit(reshaped_mfccs, data['age']).feature_importances_*100), key=lambda x: x[1], reverse=True):
    print(col, '\t', imp)

201 	 1.0002899257826312
127 	 0.6851541290052747
195 	 0.6378358346196411
235 	 0.6370599910566298
189 	 0.6210777367783331
75 	 0.6179715611684038
211 	 0.5994434313727769
89 	 0.5851780757565556
249 	 0.5542294746435765
217 	 0.5505523511568658
79 	 0.5493339438494794
0 	 0.536654923821151
69 	 0.5309325690147406
272 	 0.5243477411578349
196 	 0.5237055223760564
118 	 0.5197279857107275
136 	 0.5178547364418885
83 	 0.5137317331228208
267 	 0.5104032678227961
84 	 0.5100935574913308
287 	 0.5087244159962194
66 	 0.5011055692286573
182 	 0.49774523303747636
221 	 0.4975280477154936
285 	 0.4940234508807234
70 	 0.49032747475715177
241 	 0.48935728704380943
92 	 0.47803903344286436
86 	 0.4677853415263154
37 	 0.4589230271141162
187 	 0.45579137412937265
179 	 0.4534785391324005
94 	 0.4531276943827125
72 	 0.45288422299763775
57 	 0.4515708956937492
13 	 0.44624865751737947
183 	 0.4436027641334228
68 	 0.44273270922899344
107 	 0.4379407963869428
218 	 0.43370514034014584
112 	 0.43

In [93]:
cross_val_score(RandomForestRegressor(n_jobs=-1), reshaped_mfccs, data['age'], cv=5, scoring='neg_mean_absolute_error').mean()

np.float64(-10.549830251292219)

## Esperimenti random

In [114]:
sentence_feat = data.loc[:,['num_words','num_characters']].values
uniq, ind, count= np.unique(np.array(sentence_feat), return_counts=True, return_inverse=True, axis=0)
display(count)

array([ 409,    2,    1,   39,    8,    5,    2,    3,    3,    3,    1,
          8,    5,   19,    9,   16,    7,    7,    1,    4,    1,    2,
          9,    8,   27,   66,   26,    6,    5,    7,    2,   58,    1,
         52,   56,    6,   51,    5,    9,    4,    8,    4,    1,    2,
          3,    3,   15,    1,    6,    2,   28,    3,   55,   28,    3,
          1,   44,    1,    1,    5,    4,    5,    1,    1,    1,    2,
          6,    3,   11,   16,    1,    1,    1,    1,    1,    1, 1710])

In [161]:
print(uniq[-1])

[ 69 281]


In [145]:
long_data = data[ind==76]
display(long_data)

,sampling_rate,age,gender,ethnicity,mean_pitch,max_pitch,min_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,num_words,num_characters,num_pauses,silence_duration
path,,,,,,,,,,,,,,,,,,
1.wav,22050,24.0,female,arabic,1821.69060,3999.7170,145.43066,0.013795,0.082725,0.002254,0.210093,3112.257251,[151.99908088],-123.999726,69,281,39,23.846893
2.wav,22050,22.5,female,hungarian,1297.81870,3998.8590,145.37268,0.025349,0.096242,0.007819,0.078849,1688.016389,[129.19921875],-86.928478,69,281,21,19.388662
3.wav,22050,22.0,female,portuguese,1332.85240,3998.8025,145.42395,0.019067,0.119456,0.002974,0.105365,2576.901706,[117.45383523],-98.450670,69,281,1,21.640998
4.wav,22050,22.0,female,english,1430.34990,3998.4510,147.98083,0.017004,0.102389,0.022371,0.173701,3269.751413,[117.45383523],-56.459762,69,281,9,19.644127
5.wav,22050,22.0,male,dutch,1688.72340,3998.6113,145.44772,0.028027,0.124831,0.005369,0.107279,1930.897375,[112.34714674],-80.349204,69,281,11,18.041905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2924.wav,22050,27.0,female,czech,817.82214,3998.9214,145.37355,0.029751,0.121434,0.009271,0.077740,1583.356491,[112.34714674],-89.199463,69,281,1,24.215828
2926.wav,22050,22.0,female,albanian,1457.88110,3999.0474,145.45604,0.015260,0.087883,0.003336,0.135405,2005.046430,[92.28515625],-108.632687,69,281,18,17.925805
2928.wav,22050,39.0,female,fijian,1122.14280,3999.1540,145.44153,0.027051,0.126792,0.008059,0.111901,1861.406972,[103.359375],-67.774090,69,281,1,29.892472


In [146]:
short_data = data[ind!=76]
display(short_data)

,sampling_rate,age,gender,ethnicity,mean_pitch,max_pitch,min_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,num_words,num_characters,num_pauses,silence_duration
path,,,,,,,,,,,,,,,,,,
8.wav,22050,18.0,female,igbo,1042.95260,3989.9595,147.18710,0.013859,0.104579,0.000908,0.111858,2701.802811,[215.33203125],5.586937,4,16,1,1.367937
9.wav,22050,18.0,female,igbo,779.33765,2927.2144,157.62047,0.013921,0.080848,0.000324,0.039097,1448.093479,[143.5546875],-43.823950,0,0,1,1.989660
10.wav,22050,18.0,male,igbo,732.35297,3988.0715,147.95331,0.017735,0.114380,0.007425,0.073166,1722.624247,[86.1328125],-47.044540,5,22,1,1.437914
14.wav,22050,20.0,female,igbo,1470.73180,3994.0364,147.56389,0.012966,0.064988,0.002256,0.076528,2186.139030,[92.28515625],-16.998423,4,12,1,1.093379
15.wav,22050,20.0,male,igbo,821.28864,3533.2290,150.64523,0.016953,0.107844,0.003589,0.043305,1833.282572,[107.66601562],-60.091181,0,0,1,0.668345
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925.wav,22050,18.0,male,igbo,495.67523,1235.3107,148.78967,0.050740,0.170841,0.004776,0.022993,692.116731,[135.99917763],-30.340135,0,0,1,1.529025
2927.wav,22050,32.0,male,igbo,588.56006,3247.9130,146.66269,0.015273,0.073947,0.025475,0.048158,2039.953197,[234.90767045],-48.841359,4,15,2,3.041814
2930.wav,22050,15.0,female,igbo,1089.60050,3984.6550,145.58409,0.015317,0.126740,0.000339,0.070508,2712.362323,[83.35433468],6.757283,0,0,1,1.509206


## Features

In [147]:
data.loc[ind==76,'ethnicity'] = 1
data.loc[ind!=76,'ethnicity'] = 0
clean_data = data.drop(columns=['num_words','num_characters','num_pauses','sampling_rate','max_pitch','min_pitch'])
clean_data['tempo'] = clean_data['tempo'].map(lambda x: float(x[1:-1]))
display(clean_data)

,age,gender,ethnicity,mean_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,silence_duration
path,,,,,,,,,,,,
1.wav,24.0,female,1,1821.69060,0.013795,0.082725,0.002254,0.210093,3112.257251,151.999081,-123.999726,23.846893
2.wav,22.5,female,1,1297.81870,0.025349,0.096242,0.007819,0.078849,1688.016389,129.199219,-86.928478,19.388662
3.wav,22.0,female,1,1332.85240,0.019067,0.119456,0.002974,0.105365,2576.901706,117.453835,-98.450670,21.640998
4.wav,22.0,female,1,1430.34990,0.017004,0.102389,0.022371,0.173701,3269.751413,117.453835,-56.459762,19.644127
5.wav,22.0,male,1,1688.72340,0.028027,0.124831,0.005369,0.107279,1930.897375,112.347147,-80.349204,18.041905
...,...,...,...,...,...,...,...,...,...,...,...,...
2929.wav,24.0,male,1,1641.14930,0.023647,0.115361,0.001879,0.111799,2188.853478,184.570312,-100.921055,17.461406
2930.wav,15.0,female,0,1089.60050,0.015317,0.126740,0.000339,0.070508,2712.362323,83.354335,6.757283,1.509206
2931.wav,17.0,female,0,994.46484,0.009677,0.103535,0.001464,0.058442,2248.698477,89.102909,-53.913449,1.645034


In [149]:
full_data = pd.concat([clean_data, reshaped_mfccs], axis=1)
display(full_data)

,age,gender,ethnicity,mean_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,...,290,291,292,293,294,295,296,297,298,299
1.wav,24.0,female,1,1821.69060,0.013795,0.082725,0.002254,0.210093,3112.257251,151.999081,...,-1.942417,-5.246557,-4.376464,0.120627,-5.158453,-1.831522,-3.805745,-3.881568,-4.000273,-8.090446
2.wav,22.5,female,1,1297.81870,0.025349,0.096242,0.007819,0.078849,1688.016389,129.199219,...,1.602238,-4.590589,1.433791,0.820234,2.581253,2.410373,-4.295688,4.829690,-2.975813,0.896594
3.wav,22.0,female,1,1332.85240,0.019067,0.119456,0.002974,0.105365,2576.901706,117.453835,...,3.076923,-4.109672,-0.462944,1.148665,2.150095,-0.915860,-1.507109,2.374164,4.178719,3.131405
4.wav,22.0,female,1,1430.34990,0.017004,0.102389,0.022371,0.173701,3269.751413,117.453835,...,-5.853500,-16.481180,-12.324727,-15.200755,-9.736434,-11.306305,-17.313793,-11.241422,-13.679470,-18.686422
5.wav,22.0,male,1,1688.72340,0.028027,0.124831,0.005369,0.107279,1930.897375,112.347147,...,6.182858,-0.351926,5.084396,8.022604,6.370649,2.164866,6.037231,3.468850,5.418577,9.559019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2929.wav,24.0,male,1,1641.14930,0.023647,0.115361,0.001879,0.111799,2188.853478,184.570312,...,2.097236,-0.667882,2.911166,3.991309,3.089388,0.143708,1.004856,-3.498344,0.469718,0.205209
2930.wav,15.0,female,0,1089.60050,0.015317,0.126740,0.000339,0.070508,2712.362323,83.354335,...,-1.984530,4.366840,-2.148508,-1.667701,2.311760,1.714393,6.971056,3.783729,-4.724131,-4.516420
2931.wav,17.0,female,0,994.46484,0.009677,0.103535,0.001464,0.058442,2248.698477,89.102909,...,-4.999870,-11.469429,-10.364605,-11.625636,-0.410724,-13.186292,-10.333710,-2.208193,-1.970908,-3.222235
2932.wav,18.0,male,1,1600.00820,0.019571,0.100946,0.004451,0.115139,1834.596924,143.554688,...,5.136696,2.013695,2.777900,-0.552682,-0.650282,-4.115758,0.530575,-2.299961,-3.592226,1.672502


In [150]:
female_full_data = full_data[full_data['gender']=='female'].drop(columns=['gender'])
male_full_data = full_data[full_data['gender']=='male'].drop(columns=['gender'])
display(female_full_data)
display(male_full_data)

,age,ethnicity,mean_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,...,290,291,292,293,294,295,296,297,298,299
1.wav,24.0,1,1821.69060,0.013795,0.082725,0.002254,0.210093,3112.257251,151.999081,-123.999726,...,-1.942417,-5.246557,-4.376464,0.120627,-5.158453,-1.831522,-3.805745,-3.881568,-4.000273,-8.090446
2.wav,22.5,1,1297.81870,0.025349,0.096242,0.007819,0.078849,1688.016389,129.199219,-86.928478,...,1.602238,-4.590589,1.433791,0.820234,2.581253,2.410373,-4.295688,4.829690,-2.975813,0.896594
3.wav,22.0,1,1332.85240,0.019067,0.119456,0.002974,0.105365,2576.901706,117.453835,-98.450670,...,3.076923,-4.109672,-0.462944,1.148665,2.150095,-0.915860,-1.507109,2.374164,4.178719,3.131405
4.wav,22.0,1,1430.34990,0.017004,0.102389,0.022371,0.173701,3269.751413,117.453835,-56.459762,...,-5.853500,-16.481180,-12.324727,-15.200755,-9.736434,-11.306305,-17.313793,-11.241422,-13.679470,-18.686422
6.wav,33.0,1,1258.52160,0.025642,0.093994,0.014001,0.152036,2350.747577,123.046875,-93.453230,...,-8.174415,-8.913464,-10.072395,-13.084078,-4.271658,-6.493922,-6.987832,-6.430759,-6.534220,-4.162319
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2926.wav,22.0,1,1457.88110,0.015260,0.087883,0.003336,0.135405,2005.046430,92.285156,-108.632687,...,0.226774,2.290264,-2.824454,2.356261,1.796209,1.618824,-0.084660,-1.098554,-2.280983,-0.979584
2928.wav,39.0,1,1122.14280,0.027051,0.126792,0.008059,0.111901,1861.406972,103.359375,-67.774090,...,5.125805,4.725461,3.795096,7.323720,7.526706,3.167778,1.359615,-0.664699,6.014208,5.774710
2930.wav,15.0,0,1089.60050,0.015317,0.126740,0.000339,0.070508,2712.362323,83.354335,6.757283,...,-1.984530,4.366840,-2.148508,-1.667701,2.311760,1.714393,6.971056,3.783729,-4.724131,-4.516420
2931.wav,17.0,0,994.46484,0.009677,0.103535,0.001464,0.058442,2248.698477,89.102909,-53.913449,...,-4.999870,-11.469429,-10.364605,-11.625636,-0.410724,-13.186292,-10.333710,-2.208193,-1.970908,-3.222235


,age,ethnicity,mean_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,...,290,291,292,293,294,295,296,297,298,299
5.wav,22.0,1,1688.72340,0.028027,0.124831,0.005369,0.107279,1930.897375,112.347147,-80.349204,...,6.182858,-0.351926,5.084396,8.022604,6.370649,2.164866,6.037231,3.468850,5.418577,9.559019
7.wav,18.0,1,1789.02590,0.022059,0.138322,0.001402,0.177713,2771.793237,92.285156,-132.864988,...,-1.061666,-5.589748,-3.439590,0.449181,-0.561726,-3.568505,-0.205955,-1.555035,-2.410590,-3.159631
10.wav,18.0,0,732.35297,0.017735,0.114380,0.007425,0.073166,1722.624247,86.132812,-47.044540,...,4.086848,8.919844,12.513921,-1.471938,-5.042432,-2.831152,-1.790807,0.321547,1.039343,-6.841827
11.wav,28.0,1,1111.25890,0.021592,0.102127,0.005962,0.080557,1822.927444,123.046875,-84.304087,...,-2.362653,-4.836688,-3.237507,0.298921,-0.370968,-3.000236,-3.164778,-0.859076,-7.226579,-2.283674
12.wav,20.0,1,1331.60960,0.015685,0.104188,0.004787,0.141055,2297.955649,117.453835,-86.256958,...,-7.345084,1.630896,2.389131,1.143206,-1.852906,-0.740636,4.637617,-3.854735,6.035574,-6.804972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2923.wav,22.0,1,1732.70870,0.019848,0.114089,0.003877,0.186958,2832.177810,172.265625,-94.335132,...,-18.264940,-12.389527,-12.179855,-13.496652,-14.648852,-10.948323,-13.115424,-16.638103,-11.601784,-9.497545
2925.wav,18.0,0,495.67523,0.050740,0.170841,0.004776,0.022993,692.116731,135.999178,-30.340135,...,-13.244773,1.092194,-12.603058,-17.293257,-13.418323,-16.404589,-22.299887,-20.639952,-9.139045,-3.552170
2927.wav,32.0,0,588.56006,0.015273,0.073947,0.025475,0.048158,2039.953197,234.907670,-48.841359,...,-2.113768,-6.372054,-11.992163,3.262146,-8.659460,-4.427877,-1.516657,-7.313030,-14.316506,-11.982656
2929.wav,24.0,1,1641.14930,0.023647,0.115361,0.001879,0.111799,2188.853478,184.570312,-100.921055,...,2.097236,-0.667882,2.911166,3.991309,3.089388,0.143708,1.004856,-3.498344,0.469718,0.205209


## Modello female

In [ ]:
female = female_full_data.drop(columns=['age'])
female.columns = female.columns.astype(str)
female_age = female_full_data['age']
for col,imp in sorted(zip(female.columns, RandomForestRegressor(n_jobs=-1).fit(female, female_age).feature_importances_*100), key=lambda x: x[1], reverse=True):
    print(col, '\t', imp)

silence_duration 	 30.976482907357152
hnr 	 1.9092025567546393
shimmer 	 1.0969811815695811
128 	 1.0592713544892232
83 	 0.8746076655902851
67 	 0.7254095977085506
71 	 0.6512736424909275
170 	 0.6510293125124328
178 	 0.6119758250197623
84 	 0.5723653655715686
239 	 0.550670883083401
135 	 0.5225951257544973
165 	 0.5178851582734602
246 	 0.5042248212044818
22 	 0.49980812564145055
64 	 0.48935307566054215
162 	 0.48478823904035534
164 	 0.47430239558924137
jitter 	 0.45328522257982484
89 	 0.4468523505094188
189 	 0.445869515117981
122 	 0.4441056720757764
151 	 0.4429046801878135
5 	 0.440703606920438
58 	 0.4392472864260792
127 	 0.43600244242628816
140 	 0.43480330571629017
160 	 0.4327839358675264
23 	 0.42526190920670387
11 	 0.4108655403839528
228 	 0.40485708229718415
zcr_mean 	 0.40194710118348587
208 	 0.3997247586252061
123 	 0.39935236750552683
209 	 0.376349216479161
267 	 0.3642899196356134
37 	 0.3601382399627516
276 	 0.35903440831864336
0 	 0.3472027135684856
124 	 0

In [153]:
cross_val_score(RandomForestRegressor(n_jobs=-1), female, female_age, cv=10, scoring='neg_mean_absolute_error').mean()

np.float64(-7.384200214332308)

## Modello male

In [154]:
male = male_full_data.drop(columns=['age'])
male.columns = male.columns.astype(str)
male_age = male_full_data['age']
for col,imp in sorted(zip(male.columns, RandomForestRegressor(n_jobs=-1).fit(male, male_age).feature_importances_*100), key=lambda x: x[1], reverse=True):
    print(col, '\t', imp)

silence_duration 	 22.217247700433912
121 	 1.655527569636188
183 	 1.4933259853015912
100 	 1.318286952007992
hnr 	 1.2249401595452882
40 	 0.9171493337123302
jitter 	 0.8472681910209544
146 	 0.7786669200119974
94 	 0.7462382604154416
148 	 0.6982977221113095
190 	 0.6249264280612189
120 	 0.5898414404647836
86 	 0.5738555072929864
196 	 0.5526358462896305
141 	 0.5436471604431379
130 	 0.5279344148536391
151 	 0.5222841768221841
8 	 0.5021957027586691
139 	 0.49189952651725943
44 	 0.4917477111230456
264 	 0.481176331749483
mean_pitch 	 0.4601084564063069
93 	 0.4403577784422246
135 	 0.43097919542108554
262 	 0.4301818563604901
17 	 0.4271651639826399
235 	 0.42163908181202975
81 	 0.4139857040607989
265 	 0.4101019124618794
89 	 0.40972774839082976
156 	 0.4071865218956025
169 	 0.40567861047827297
229 	 0.40475568851814725
3 	 0.39947196820334296
shimmer 	 0.3991288269389981
132 	 0.3934536034966219
54 	 0.38772233449510946
124 	 0.36715498024582893
158 	 0.36343338922226187
46 	

In [155]:
cross_val_score(RandomForestRegressor(n_jobs=-1), male, male_age, cv=10, scoring='neg_mean_absolute_error').mean()

np.float64(-7.653621046500791)

## Consegna risposte

In [205]:
data_evaluation = pd.read_csv('../data/evaluation.csv', index_col=0)
data_evaluation['path'] = data_evaluation['path'].map(lambda x: x.split('/')[1]) # int(x.split('/')[1].split('.')[0]))
display(data_evaluation)

,sampling_rate,gender,ethnicity,mean_pitch,max_pitch,min_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,num_words,num_characters,num_pauses,silence_duration,path
Id,,,,,,,,,,,,,,,,,,
0,22050,male,spanish,1056.91740,3945.1610,145.38750,0.022082,0.171076,0.003136,0.032963,1549.607050,[80.74951172],-116.662338,69,281,2,38.198503,1.wav
1,22050,male,xiang,1231.84570,3999.1720,145.56432,0.026571,0.132585,0.006783,0.123895,2344.817369,[89.10290948],-78.253824,69,281,27,29.605442,2.wav
2,22050,male,igbo,958.29065,3445.4490,145.67374,0.018044,0.096289,0.004478,0.089149,1939.574896,[123.046875],-71.630742,6,22,2,2.275556,3.wav
3,22050,female,spanish,1396.54170,3998.8948,145.41223,0.027290,0.088901,0.014893,0.097054,1832.059113,[123.046875],-101.533013,69,281,31,22.151837,4.wav
4,22050,male,spanish,1633.86770,3999.7632,145.36313,0.021621,0.103855,0.001369,0.140950,2534.611168,[112.34714674],-134.914070,69,281,35,22.430476,5.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
686,22050,male,igbo,570.62740,3900.6730,145.67577,0.018842,0.079197,0.004545,0.072083,1946.502158,[112.34714674],-42.895295,6,15,1,1.861950,687.wav
687,22050,male,igbo,974.13965,3919.0024,145.90408,0.024367,0.117492,0.000878,0.076900,3319.620800,[112.34714674],-144.881089,7,22,1,4.876190,688.wav
688,22050,female,serbian,1113.27650,3999.3510,145.38307,0.020637,0.089355,0.009148,0.095613,1973.127197,[112.34714674],-73.559944,69,281,15,25.541950,689.wav


In [206]:
np.unique(data_evaluation['sampling_rate'].values)

array([22050])

In [207]:
SAMPLE_RATE = 22050
num_time_buckets = 20

In [200]:
audio_files_evaluation = os.listdir('../data/audios_evaluation')
audio_arrays_evaluation,mfccs_evaluation,time_sampl_evaluation = load_data(audio_files_evaluation, '../data/audios_evaluation/', SAMPLE_RATE)

In [208]:
standardized_mfccs_evaluation = standardize_all(mfccs_evaluation, num_time_buckets)
reshaped_mfccs_evaluation = reshape_all(standardized_mfccs_evaluation)

In [209]:
display(reshaped_mfccs_evaluation)

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
1.wav,-409.024384,-318.300507,-356.190277,-345.002319,-337.278931,-334.057831,-357.991608,-326.944092,-306.628845,-350.886414,...,3.988764,1.971522,4.218048,2.856964,0.434794,1.786844,4.884138,7.342025,6.462479,1.600923
10.wav,-435.284241,-392.018524,-248.948151,-323.007416,-292.441101,-270.305969,-284.768127,-329.105835,-327.972534,-353.197693,...,-5.979064,-5.135306,3.936126,0.141418,-5.740375,-13.966771,-5.430674,-2.823639,-10.169382,-2.961752
100.wav,-293.302338,-247.493042,-293.399414,-261.878693,-259.596832,-272.239227,-267.624451,-311.161499,-248.529968,-275.995758,...,14.765183,14.073303,12.891003,8.772925,11.083576,10.430563,14.053826,11.543507,11.924887,10.976063
101.wav,-193.993469,-331.810791,-270.091370,-290.040497,-326.564789,-264.394196,-321.724152,-238.821533,-321.110321,-241.872086,...,8.019660,7.337913,9.528759,6.420187,6.243932,4.817814,8.012930,5.655083,5.977262,7.757629
102.wav,-431.241058,-335.702271,-277.690338,-265.180511,-357.530884,-275.078308,-324.168732,-250.916199,-342.306335,-235.404755,...,-13.446664,-14.555564,-13.896852,-6.870616,-10.348890,-13.119678,-8.803543,-8.120754,-8.011740,-12.725560
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95.wav,-519.660828,-312.015198,-345.898254,-301.076508,-305.187042,-230.281097,-357.826508,-327.026001,-359.214020,-415.862976,...,0.992612,7.384030,1.896030,-11.600630,-2.917705,6.524923,6.019863,13.066234,7.906710,11.046993
96.wav,-335.466797,-317.808655,-165.252213,-214.704208,-170.736435,-202.559998,-216.286987,-148.046417,-180.815384,-172.436188,...,0.697124,4.973277,-6.263444,-3.027616,0.056703,2.821440,-2.695671,1.973516,3.052486,0.571430
97.wav,-503.112793,-443.052155,-453.366699,-453.388092,-230.930328,-284.111115,-296.293030,-288.297302,-323.740601,-304.597900,...,7.643392,-8.778545,-5.423125,-2.170265,-9.004732,-16.067131,-6.832082,0.283662,-1.104228,0.164042
98.wav,-261.113708,-330.292542,-284.257782,-275.578186,-321.897217,-259.656433,-314.730774,-286.421936,-298.456909,-323.157501,...,0.490386,3.241124,-0.128632,4.476760,8.151598,0.362155,2.285372,4.306463,8.140712,1.063711


In [204]:
temp = reshaped_mfccs_evaluation.reset_index()
temp['index'] = temp['index'].map(lambda x: int(x.split('.')[0])-1)
reshaped_mfccs_evaluation_ind = temp.set_index('index').sort_index()
display(reshaped_mfccs_evaluation_ind)

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
index,,,,,,,,,,,,,,,,,,,,,
0,-409.024384,-318.300507,-356.190277,-345.002319,-337.278931,-334.057831,-357.991608,-326.944092,-306.628845,-350.886414,...,3.988764,1.971522,4.218048,2.856964,0.434794,1.786844,4.884138,7.342025,6.462479,1.600923
1,-336.617706,-261.725800,-240.710938,-344.066650,-297.420715,-252.167053,-258.892456,-270.511047,-272.087616,-254.859573,...,5.123708,5.566287,8.486043,4.639581,5.483193,0.815799,1.230392,4.926414,3.831308,4.204085
2,-445.425537,-434.862152,-305.619904,-194.119293,-183.565063,-196.765488,-183.358307,-205.783173,-259.249573,-177.744232,...,2.637795,-2.742330,-8.127054,4.732387,-2.522875,-0.810776,-1.422979,3.038233,7.989042,-7.020223
3,-313.318878,-274.385010,-246.486557,-273.996124,-306.219360,-255.408829,-288.595795,-251.849548,-294.436462,-270.199249,...,-6.895550,-2.852007,-1.263843,-2.992028,-2.460367,-3.775903,-2.890278,-1.230701,-2.002275,-4.992485
4,-377.444305,-336.198242,-347.945862,-377.295685,-384.000092,-390.826843,-382.165619,-291.606873,-362.531250,-321.496216,...,-2.559943,-1.841891,-3.630640,-0.582794,-4.660622,-2.600523,0.338119,3.084750,-0.355871,-0.955362
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
686,-525.936646,-463.163910,-285.372253,-173.060684,-185.194565,-265.145233,-264.845703,-319.030701,-299.257751,-263.070770,...,-14.276779,-11.735075,-6.766476,-9.147609,-13.228241,-9.561797,-3.902442,1.958264,-1.463599,-5.913436
687,-478.394135,-451.433533,-442.786377,-448.233002,-445.452393,-452.258240,-444.285889,-450.208344,-451.439697,-339.876648,...,-11.725979,-12.265416,1.553086,-2.849491,0.542458,3.470596,-0.493849,-7.555924,-2.488146,8.529310
688,-271.483917,-239.948807,-328.058838,-259.273132,-248.833664,-284.503174,-261.941864,-278.541779,-265.497742,-304.869049,...,-4.667590,-8.239069,-4.484377,-1.738832,-3.900306,-2.096578,-5.937396,-2.954125,-8.934360,-4.939095


In [210]:
data_evaluation.loc[data_evaluation['num_characters']==281,'ethnicity'] = 1 # and data_evaluation['num_words']==69),'ethnicity'] = 1
data_evaluation.loc[data_evaluation['num_characters']!=281,'ethnicity'] = 0 # or data_evaluation['num_words']!=69),'ethnicity'] = 0
clean_data_evaluation = data_evaluation.drop(columns=['num_words','num_characters','num_pauses','sampling_rate','max_pitch','min_pitch'])
clean_data_evaluation['tempo'] = clean_data_evaluation['tempo'].map(lambda x: float(x[1:-1]))
display(clean_data_evaluation)

,gender,ethnicity,mean_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,silence_duration,path
Id,,,,,,,,,,,,
0,male,1,1056.91740,0.022082,0.171076,0.003136,0.032963,1549.607050,80.749512,-116.662338,38.198503,1.wav
1,male,1,1231.84570,0.026571,0.132585,0.006783,0.123895,2344.817369,89.102909,-78.253824,29.605442,2.wav
2,male,0,958.29065,0.018044,0.096289,0.004478,0.089149,1939.574896,123.046875,-71.630742,2.275556,3.wav
3,female,1,1396.54170,0.027290,0.088901,0.014893,0.097054,1832.059113,123.046875,-101.533013,22.151837,4.wav
4,male,1,1633.86770,0.021621,0.103855,0.001369,0.140950,2534.611168,112.347147,-134.914070,22.430476,5.wav
...,...,...,...,...,...,...,...,...,...,...,...,...
686,male,0,570.62740,0.018842,0.079197,0.004545,0.072083,1946.502158,112.347147,-42.895295,1.861950,687.wav
687,male,0,974.13965,0.024367,0.117492,0.000878,0.076900,3319.620800,112.347147,-144.881089,4.876190,688.wav
688,female,1,1113.27650,0.020637,0.089355,0.009148,0.095613,1973.127197,112.347147,-73.559944,25.541950,689.wav


In [214]:
joined_data_evaluation = clean_data_evaluation.join(reshaped_mfccs_evaluation, on='path').drop(columns=['path'])

In [190]:
full_data_evaluation = pd.concat([clean_data_evaluation, reshaped_mfccs_evaluation_ind], axis=1)
display(full_data_evaluation)

,gender,ethnicity,mean_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,...,290,291,292,293,294,295,296,297,298,299
1,male,1,1056.91740,0.022082,0.171076,0.003136,0.032963,1549.607050,80.749512,-116.662338,...,3.988764,1.971522,4.218048,2.856964,0.434794,1.786844,4.884138,7.342025,6.462479,1.600923
2,male,1,1231.84570,0.026571,0.132585,0.006783,0.123895,2344.817369,89.102909,-78.253824,...,5.123708,5.566287,8.486043,4.639581,5.483193,0.815799,1.230392,4.926414,3.831308,4.204085
3,male,0,958.29065,0.018044,0.096289,0.004478,0.089149,1939.574896,123.046875,-71.630742,...,2.637795,-2.742330,-8.127054,4.732387,-2.522875,-0.810776,-1.422979,3.038233,7.989042,-7.020223
4,female,1,1396.54170,0.027290,0.088901,0.014893,0.097054,1832.059113,123.046875,-101.533013,...,-6.895550,-2.852007,-1.263843,-2.992028,-2.460367,-3.775903,-2.890278,-1.230701,-2.002275,-4.992485
5,male,1,1633.86770,0.021621,0.103855,0.001369,0.140950,2534.611168,112.347147,-134.914070,...,-2.559943,-1.841891,-3.630640,-0.582794,-4.660622,-2.600523,0.338119,3.084750,-0.355871,-0.955362
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
687,male,0,570.62740,0.018842,0.079197,0.004545,0.072083,1946.502158,112.347147,-42.895295,...,-14.276779,-11.735075,-6.766476,-9.147609,-13.228241,-9.561797,-3.902442,1.958264,-1.463599,-5.913436
688,male,0,974.13965,0.024367,0.117492,0.000878,0.076900,3319.620800,112.347147,-144.881089,...,-11.725979,-12.265416,1.553086,-2.849491,0.542458,3.470596,-0.493849,-7.555924,-2.488146,8.529310
689,female,1,1113.27650,0.020637,0.089355,0.009148,0.095613,1973.127197,112.347147,-73.559944,...,-4.667590,-8.239069,-4.484377,-1.738832,-3.900306,-2.096578,-5.937396,-2.954125,-8.934360,-4.939095
690,male,1,1759.17420,0.026118,0.106429,0.003707,0.141474,2137.517812,117.453835,-93.562873,...,-3.350654,-4.871803,-2.361799,-2.318775,-3.458122,-8.727138,-4.239640,-2.740751,-1.829307,-3.902994


In [191]:
female_full_data_evaluation = full_data_evaluation[full_data_evaluation['gender']=='female'].drop(columns=['gender'])
male_full_data_evaluation = full_data_evaluation[full_data_evaluation['gender']=='male'].drop(columns=['gender'])
female_full_data_evaluation.columns = female_full_data_evaluation.columns.astype(str)
male_full_data_evaluation.columns = male_full_data_evaluation.columns.astype(str)

In [192]:
ypred_female = RandomForestRegressor(n_jobs=-1).fit(female, female_age).predict(female_full_data_evaluation)
ypred_male = RandomForestRegressor(n_jobs=-1).fit(male, male_age).predict(male_full_data_evaluation)

In [193]:
ypred_female_series = pd.Series(ypred_female, name='Predicted', index=female_full_data_evaluation.index)
ypred_male_series = pd.Series(ypred_male, name='Predicted', index=male_full_data_evaluation.index)
ypred = pd.concat([ypred_female_series, ypred_male_series]).sort_index()
display(ypred)

1      29.770
2      29.230
3      22.140
4      31.540
5      34.080
        ...  
687    20.690
688    25.000
689    27.365
690    41.330
691    19.700
Name: Predicted, Length: 690, dtype: float64

In [196]:
pd.DataFrame(ypred, index=data_evaluation.index).to_csv('submissionMaddalena.csv') # hits 10.367 above naive baseline